# Sprint 3 — Process Mining
Análise de processos hospitalares com PM4Py sobre o `gold_event_log`.

**Notebook:** `03_process_mining.ipynb`  
**Pipeline de origem:** `gold_transformations`  
**Schema de leitura:** `hospital_santa_rosa.gold_fluxo`  
**Autor:** Ediney Magalhães  
**Criado em:** 2026-06-15

In [0]:
%pip install pm4py
dbutils.library.restartPython()

In [0]:
import pm4py
import pandas as pd

In [0]:
# leitura do event log canônico fo Unity Catalog como Spark DataFrame
df_spark = spark.table("hospital_santa_rosa.gold_fluxo.gold_event_log")

# confirma volume e schema antes de trazer para memória
print(f'Total de registros: {df_spark.count():,}')
df_spark.printSchema()

In [0]:
# converte Spark DataFrame para Pandas
df_pandas = df_spark.toPandas()

# confirma que a conversão manteve o volume correto
print(f'Registros no Pandas: {len(df_pandas):,}')
print(f'Tipo timestamp: {df_pandas['timestamp'].dtype}')

In [0]:
# declarando timestamp em UTC (PM4PY necessita de timestamp com timezone explícito)
df_pandas['timestamp'] = df_pandas['timestamp'].dt.tz_localize('UTC')

# confirmação do tipo correto
print(f'Tipo timestamp após correção: {df_pandas['timestamp'].dtype}')

In [0]:
# preparando DataFrame para PM4PY, declarando quais colunas corresponde ao conceito XES
df_formatado = pm4py.format_dataframe(
    df_pandas,
    case_id='case_id',
    activity_key='activity',
    timestamp_key='timestamp'
)

# confirmando aplicação 
print(f'Colunas após format_dataframe: {list(df_formatado.columns)}')

In [0]:
# converte o DataFrame pandas anotado para o objeto EventLog hierárquico PM4Py
event_log = pm4py.convert_to_event_log(df_formatado)

# confirma o número de traces (casos) e eventos no EventLog
print(f"Total de traces: {len(event_log):,}")
print(f"Total de eventos: {sum(len(trace) for trace in event_log):,}")

## Descoberta de Processo

In [0]:
# aplicando o algoritmo Inductive Miner
process_tree = pm4py.discover_process_tree_inductive(event_log)

## Variant Analysis

In [0]:
variantes = pm4py.get_variants(event_log)
print(f"Total de variantes distintas: {len(variantes)}")

In [0]:
# converte o dicionário de variantes para uma lista ranqueada por frequência
ranking_variantes = sorted(
    variantes.items(),
    key=lambda item: len(item[1]),
    reverse=True
)

# exibe as 10 variantes mais frequentes
total_casos = len(event_log)
print(f"{'Rank':<6} {'Casos':>6} {'% do Total':>10}  Sequência")
print("-" * 80)
for rank, (sequencia, traces) in enumerate(ranking_variantes[:10], start=1):
    frequencia = len(traces)
    percentual = frequencia / total_casos * 100
    atividades = " → ".join(sequencia)
    print(f"{rank:<6} {frequencia:>6} {percentual:>9.1f}%  {atividades}")

In [0]:
import pandas as pd

# constrói o DataFrame do ranking de variantes
linhas = []
for rank, (sequencia, traces) in enumerate(ranking_variantes, start=1):
    linhas.append({
        "rank": rank,
        "sequencia": " → ".join(sequencia),
        "total_eventos": len(sequencia),
        "total_casos": len(traces),
        "cobertura_perc": round(len(traces) / total_casos * 100, 2),
        "data_referencia": pd.Timestamp.now(tz="UTC").date()
    })

df_variantes = pd.DataFrame(linhas)

# persiste o ranking de variantes no schema gold_fluxo
df_spark_variantes = spark.createDataFrame(df_variantes)

df_spark_variantes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("hospital_santa_rosa.gold_fluxo.gold_variant_analysis")

print(f"Total de variantes: {len(df_variantes):,}")
print("gold_variant_analysis salva com sucesso")

## Bottleneck Detection

In [0]:
# calcula o bottleneck para cada fonte separadamente e empilha os resultados
dfs = []

for source in df_formatado["source"].unique():
    
    # filtra por fonte
    df_source = df_formatado[df_formatado["source"] == source].copy()
    
    # ordena por caso e timestamp
    df_source = df_source.sort_values(["case:concept:name", "time:timestamp"])
    
    # cria colunas de atividade e timestamp anteriores
    df_source["activity_anterior"] = df_source.groupby("case:concept:name")["concept:name"].shift(1)
    df_source["timestamp_anterior"] = df_source.groupby("case:concept:name")["time:timestamp"].shift(1)
    
    # remove primeiro evento de cada caso
    df_source = df_source.dropna(subset=["activity_anterior"])
    
    # calcula tempo entre atividades em minutos
    df_source["tempo_entre_atividades_min"] = (
        df_source["time:timestamp"] - df_source["timestamp_anterior"]
    ).dt.total_seconds() / 60
    
    # adiciona coluna de identificação da fonte
    df_source["source"] = source
    
    dfs.append(df_source)

# empilha todos os resultados
df_bottleneck = pd.concat(dfs, ignore_index=True)

In [0]:
# conferindo
df_bottleneck[["case:concept:name", "activity_anterior", "concept:name", "tempo_entre_atividades_min"]].head(10)

In [0]:
# agregando tempos por pares de atividades
df_bottleneck_agg = (
    df_bottleneck
    .groupby(["source","activity_anterior", "concept:name"])["tempo_entre_atividades_min"]
    .agg(["mean", "median", "std", "count"])
    .reset_index()
    .rename(columns={
        "source": "source",
        "activity_anterior": "de",
        "concept:name": "para",
        "mean": "tempo_medio_min",
        "median": "tempo_mediano_min",
        "std": "desvio_padrao_min",
        "count": "frequencia"
    })
    .sort_values("tempo_medio_min", ascending=False)
)

# adiciona coluna com cálculo do coeficiente de variação
df_bottleneck_agg["cv_pct"] = (
    df_bottleneck_agg["desvio_padrao_min"] / df_bottleneck_agg["tempo_medio_min"] * 100
).round(1)
# ordena pelos valores da média 
df_bottleneck_agg = df_bottleneck_agg.sort_values("tempo_medio_min", ascending=False)

# verificando
df_bottleneck_agg.head(10)

In [0]:
# visualizando em formato de tabela
df_bottleneck_agg[
    (df_bottleneck_agg["frequencia"] >= 30) & 
    (df_bottleneck_agg["de"] != df_bottleneck_agg["para"])
     ]

### Visualização — Top Gargalos por Tempo Mediano

In [0]:
import matplotlib.pyplot as plt

# filtra DataFrame para o gráfico (relevância para pares com frquência maior e sem auto-transições)
df_plot = df_bottleneck_agg[
    (df_bottleneck_agg["frequencia"] >= 30) &
    (df_bottleneck_agg["de"] != df_bottleneck_agg["para"]) &
    (df_bottleneck_agg["source"] == "silver_atendimento_emergencia")
].copy()

# rótulo do par de atividades
df_plot["transicao"] = df_plot["de"] + " → " + df_plot["para"]

# ordena as barras do maior tempo mediano para o menor
df_plot = df_plot.sort_values("tempo_mediano_min", ascending=True)

# construção do gráfico: cria a figura e o eixo
fig, ax = plt.subplots(figsize=(10, 6))
# barras horizontais
ax.barh(df_plot["transicao"], df_plot["tempo_mediano_min"], color="steelblue")
#  rótulo do eixo X e título do gráfico
ax.set_xlabel("Tempo Mediano (minutos)")
ax.set_title("Gargalos no Processo de Emergência — Tempo Mediano entre Atividades")
# ajuste de margens para os rótulos não ficarem cortados
plt.tight_layout()
plt.show()

## Conformance Checking

In [0]:
# converte o Process Tree para Petri Net
net, initial_marking, final_marking = pm4py.convert_to_petri_net(process_tree)

In [0]:
# lista que vai acumular os resultados de cada fonte
resultados_conformance = []

# constroi indice case_id, trace para acesso direto em O(1)
indice_traces = {
    trace.attributes["concept:name"]: trace
    for trace in event_log
}

# iterando sobre cada fonte única
for source in df_formatado["source"].unique():
    cases_source = df_formatado[
        df_formatado["source"] == source
    ]["case:concept:name"].unique().tolist()
    # filtro pelos cases da fonte
    event_log_source = pm4py.objects.log.obj.EventLog()
    # itera apenas sobre os cases daquela fonte
    for case_id in cases_source:
        # verifica se o case existe no dicionário
        if case_id in indice_traces:
            # acessa o trace pela chave e adiciona ao EventLog da fonte
            event_log_source.append(indice_traces[case_id])
    # descobre o Process Tree para cada fonte
    process_tree_source = pm4py.discover_process_tree_inductive(event_log_source)
    # converte o Process Tree para Petri Net
    net_source, im_source, fm_source = pm4py.convert_to_petri_net(process_tree_source)
    # calcula o fitness para a fonte (recebe o EventLog filtrado da fonte e os três objetos da Petri Net)
    fitness_source = pm4py.fitness_token_based_replay(
        event_log_source,
        net_source,
        im_source,
        fm_source
    )
    # calcula a precision para a fonte
    precision_source = pm4py.precision_token_based_replay(
        event_log_source,
        net_source,
        im_source,
        fm_source
    )
    # dicionário de resultado para a fonte e adiciona na lista
    resultados_conformance.append({
        "source": source,
        "fitness": round(fitness_source["log_fitness"], 4),
        "precision": round(precision_source, 4),
        "total_traces": len(event_log_source)
    })


In [0]:
df_conformance = pd.DataFrame(resultados_conformance)
print(df_conformance)

## Social Network Analysis (Organizational Mining)

Cinco análises de handover/subcontracting entre setores e especialidades.

Ator = `source` (setor/processo de origem) ou `especialidade`, conforme ADR-0008.

Ordem de implementação: nativa pura → nativa com filtro → construção manual em pandas.

### #5 — Subcontracting, Setor↔Setor (nível único, sem especialidade)

In [0]:
# calcula Subcontracting entre setores, padrão A>B>A dentro do mesmo caso
subcontracting_setor = pm4py.discover_subcontracting_network(event_log, resource_key="source")

In [0]:
# lista todos os atributos e métodos do objeto 
print(dir(subcontracting_setor))

In [0]:
# inspecionando connections
print(subcontracting_setor.connections)

**Interpretação:** o resultado do `discover_subcontracting_network` é dominado
por relações intra-setor (A, A) — artefato estrutural do padrão A→B→A quando
B=A (sequência natural de eventos dentro do mesmo `source`), sem significado
de negócio, equivalente ao `de == para` já descartado no Bottleneck Detection.

O único par inter-setor real é `(silver_internacoes, silver_movimentacoes)`,
com peso de 0.00013 — desprezível.

**Decisão:** Subcontracting setor↔setor (nível único, sem especialidade) não
revela padrão relevante nos dados — delegação-com-retorno entre setores é
rara ou inexistente, consistente com a expectativa clínica de que transições
entre setores tendem a ser definitivas, não temporárias. Não vou validar a
contagem bruta por trás dos pesos, dado o sinal desprezível.
Segue para #2 (Handover especialidade↔especialidade).

### #2 — Handover of Work, Especialidade↔Especialidade (segmentado por setor)

In [0]:
# lista de setores (source) únicos no event_log
setores = df_formatado["source"].unique().tolist()
print(setores)

In [0]:
from pm4py.algo.filtering.log.attributes import attributes_filter

# filtra o event_log mantendo só eventos do setor de teste
event_log_setor_teste = attributes_filter.apply_events(
    event_log,
    ["silver_atendimento_emergencia"],
    parameters={
        attributes_filter.Parameters.ATTRIBUTE_KEY: "source",
        attributes_filter.Parameters.POSITIVE: True
    }
)

print(f"Total de traces: {len(event_log_setor_teste)}")
print(f"Total de eventos: {sum(len(t) for t in event_log_setor_teste)}")

**Achado de qualidade de dados (não bloqueia o SNA):** o filtro por evento
confirma 46.585 eventos para `silver_atendimento_emergencia` em `df_formatado`
consistente com o `event_log` do PM4Py. Porém, uma contagem direta em SQL
contra `gold_event_log` (22/06, investigação de cobertura de `resource`)
registrou 49.888 — diferença de ~3.303 eventos.

Hipótese mais provável: descarte silencial de linhas com timestamp nulo/NaT
durante a conversão Spark > pandas > EventLog (`tz_localize` ou
`convert_to_event_log`), consistente com as lacunas de cobertura de timestamp
já documentadas em `gold_data_quality` (RQ-001–RQ-003). Não confirmado.

**Ação futura:** comparar contagem por `source` em cada etapa da pipeline de
conversão (Célula 3 vs Célula 4 vs Célula 7), para confirmar onde exatamente
a perda ocorre.

In [0]:
# lista para acumular os resultados de cada setor
resultados_handover_especialidade = []

# lista de especialidades válidas (excluindo nulos)
especialidades_validas = [
    e for e in df_formatado["especialidade"].unique() if e is not None
]

# itera sobre cada setor, calculando Handover entre especialidades dentro dele
for setor in setores:
    # filtra o event_log mantendo só eventos do setor da vez
    event_log_setor = attributes_filter.apply_events(
        event_log,
        [setor],
        parameters={
            attributes_filter.Parameters.ATTRIBUTE_KEY: "source",
            attributes_filter.Parameters.POSITIVE: True

        }    
    )
    
    # filtro para remover eventos sem especialidades válidas (excluindo nulos)
    event_log_setor = attributes_filter.apply_events(
        event_log_setor,
        especialidades_validas,
        parameters={
            attributes_filter.Parameters.ATTRIBUTE_KEY: "especialidade",
            attributes_filter.Parameters.POSITIVE: True
        }
    )

    # pula o setor se não houver eventos com especialidade válida
    if len(event_log_setor) == 0:
        print(f"Setor '{setor}' sem especialidade preenchida - pulando.")
        continue
    
    # calcula Handover of Work entre especialidades dentro deste setor
    handover_setor = pm4py.discover_handover_of_work_network(
        event_log_setor,
        resource_key="especialidade"
    )
    # guarda cada par de (especialidade A, especialidade B) com o setor de origem
    for (esp_a, esp_b), peso in handover_setor.connections.items():
        resultados_handover_especialidade.append({
            "setor": setor,
            "especialidade_a": esp_a,
            "especialidade_b": esp_b,
            "peso": peso
        })
# converte os resultados acumulados em DataFrame
df_handover_especialidade = pd.DataFrame(resultados_handover_especialidade)

print(f"Total de linhas: {len(df_handover_especialidade)}")
df_handover_especialidade.head(10)

**Interpretação:** diferente da #5, aqui A==A (mesma especialidade em sequência)
tem significado clínico real, continuidade de atendimento dentro do mesmo
setor (ex: a Clínica Médica mantém o paciente na Emergência sem repassar a
outra especialidade). Não filtra esses pares.

**Limitação estrutural:** por ser segmentada por setor, a #2 nunca captura
handover entre setores diferentes (ex: Emergência → Internação), essa
transição é capturada pela #1 (Handover setor↔setor), ainda pendente.

Dois setores foram excluídos por ausência estrutural de `especialidade`:
`silver_exames_laboratoriais` e `silver_movimentacoes`.

### #4 — Subcontracting, Especialidade↔Especialidade (segmentado por setor)

In [0]:
# lista para acumular os resultados de Subcontracting entre especialidades por setor
resultados_subcontracting_especialidade = []

# itera sobre cada setor, calculando Subcontracting entre especialidades dentro dele
for setor in setores:
    # filtra event_log mantendo só eventos do setor da vez
    event_log_setor = attributes_filter.apply_events(
        event_log,
        [setor],
        parameters={
            attributes_filter.Parameters.ATTRIBUTE_KEY: "source",
            attributes_filter.Parameters.POSITIVE: True
        }
    )
    
    # filtro para remover eventos sem especialidades válidas (excluindo nulos)
    event_log_setor = attributes_filter.apply_events(
        event_log_setor,
        especialidades_validas,
        parameters={
            attributes_filter.Parameters.ATTRIBUTE_KEY: "especialidade",
            attributes_filter.Parameters.POSITIVE: True
        }
    )

    # pula o setor se não houver eventos com especialidade válida
    if len(event_log_setor) == 0:
        print(f"Setor '{setor}' sem especialidade preenchida - pulando.")
        continue
    
    # tenta calcular Subcontracting entre especialidades dentro deste setor (se não houver padrão pula o setor)
    try:
        subcontracting_setor_esp = pm4py.discover_subcontracting_network(
            event_log_setor,
            resource_key="especialidade"
        )
    except ValueError as e:
        print(f"Setor '{setor}' sem padrão de subcontracting - pulando.")
        continue
    # guarda cada par de (especialidade A, especialidade B) com o setor de origem
    for (esp_a, esp_b), peso in subcontracting_setor_esp.connections.items():
        resultados_subcontracting_especialidade.append({
            "setor": setor,
            "especialidade_a": esp_a,
            "especialidade_b": esp_b,
            "peso": peso
        })
# converte os resultados acumulados em DataFrame
df_subcontracting_especialidade = pd.DataFrame(resultados_subcontracting_especialidade)

print(f"Total de linhas: {len(df_subcontracting_especialidade)}")
df_subcontracting_especialidade.head(10)

**Interpretação:** mesma leitura da #2 — A==A representa continuidade da
especialidade dentro do setor, não é artefato sem sentido aqui.

Três setores excluídos:
- `silver_exames_laboratoriais`, `silver_movimentacoes`, sem coluna
  `especialidade` (ausência estrutural).
- `silver_internacoes`, tem especialidade 100% preenchida, mas nenhum
  padrão A>B>A ocorre nesse setor. Consistente com a expectativa clínica:
  internação tende a ter progressão/continuidade de especialidade, não
  alternância com retorno.

In [0]:
print(df_formatado.columns.tolist())

### #1 — Handover of Work, Setor↔Setor (especialidade como atributo da transição)

In [0]:
# ordena por caso e timestamp, base para cálculo de handover
df_handover_setor = df_formatado.sort_values(["case:concept:name", "time:timestamp"]).copy()

# cria colunas com o setor e a especialidade do evento anterior, dentro do mesmo caso
df_handover_setor["source_anterior"] = df_handover_setor.groupby("case:concept:name")["source"].shift(1)
df_handover_setor["especialidade_anterior"] = df_handover_setor.groupby("case:concept:name")["especialidade"].shift(1)

# mantém só as linhas onde o setor mudou (handover real entre os setores)
df_handover_setor = df_handover_setor[
    df_handover_setor["source"] != df_handover_setor["source_anterior"]
]

# remove o início de cada caso (sem o evento anterior, não é handover real)
df_handover_setor = df_handover_setor[df_handover_setor["source_anterior"].notna()]

# agrega contando ocorrências de cada combinação setor_origem > setor_destino por especialidade
df_handover_setor_agg = (
    df_handover_setor
    .groupby(["source_anterior", "source", "especialidade"])
    .size()
    .reset_index(name="frequencia")
)

# ordena por frequência, do mais comum para o mais raro
df_handover_setor_agg = df_handover_setor_agg.sort_values("frequencia", ascending=False)

print(f"Total de combinações distintas: {len(df_handover_setor_agg)}")
df_handover_setor_agg.head(10)

**Avaliação de valor para apresentação executiva:** das cinco análises de SNA,
esta (#1 — Handover setor↔setor com especialidade na transição) é a que
revela padrão de fluxo real e interpretável para gestores. As demais (#2, #4,
#5) serviram principalmente para confirmar ausência ou trivialidade de
padrão dentro do mesmo ator, valor metodológico, não executivo. Candidata
prioritária para visualização no Sprint 4.

### #3 — Subcontracting, Setor↔Setor (especialidade como atributo da transição)

In [0]:
# ordena por caso e timestamp, base para o cálculo de Subcontracting
df_subcontracting_setor = df_formatado.sort_values(["case:concept:name", "time:timestamp"]).copy()

# cria colunas com setor e especialidade de um e dois eventos atrás, dentro do mesmo caso
df_subcontracting_setor["source_1_atras"] = df_subcontracting_setor.groupby("case:concept:name")["source"].shift(1)
df_subcontracting_setor["source_2_atras"] = df_subcontracting_setor.groupby("case:concept:name")["source"].shift(2)
df_subcontracting_setor["especialidade_1_atras"] = df_subcontracting_setor.groupby("case:concept:name")["especialidade"].shift(1)

# filtra linhas onde o padrão A>B>A ocorre (volta ao mesmo setor após um intermediário diferente)
df_subcontracting_setor = df_subcontracting_setor[
    (df_subcontracting_setor["source_2_atras"] == df_subcontracting_setor["source"]) &
    (df_subcontracting_setor["source_1_atras"] != df_subcontracting_setor["source"])
]

# agrega contando ocorrências de cada combinação setor_A > setor_B (intermediário) > volta a A, por especialidade de B
df_subcontracting_setor_agg = (
    df_subcontracting_setor
    .groupby(["source", "source_1_atras", "especialidade_1_atras"])
    .size()
    .reset_index(name="frequencia")
)

# ordena por frequência, do mais comum para o mais raro
df_subcontracting_setor_agg = df_subcontracting_setor_agg.sort_values("frequencia", ascending=False)

print(f"Total de combinações distintas: {len(df_subcontracting_setor_agg)}")
df_subcontracting_setor_agg.head(15)

In [0]:
caso_exemplo = casos_padrao_alta_cirurgia[0]

resultado = df_formatado[
    df_formatado["case:concept:name"] == caso_exemplo
][["case:concept:name", "concept:name", "time:timestamp", "source", "especialidade"]].sort_values("time:timestamp")

print(resultado.to_string(index=False))

**Achado de qualidade de dados (não corrigido nesta sessão):** o padrão
Cirurgias>Altas>Cirurgias (linhas 16 e 12 do resultado, 38 casos no total)
é causado por um evento `Alta médica` em `gold_events_altas` com timestamp
cronologicamente incoerente, ocorre **antes** do `Fim da Anestesia` do
mesmo procedimento, o que é clinicamente impossível. A alta real do caso
(`Alta Hospitalar`) tem timestamp correto e consistente com o resto da
timeline.

Hipótese: mapeamento de timestamp errado para esse evento específico em
`gold_events_altas` ou na coluna de origem na Silver. Não investigado a
fundo, fica pendente para sessão dedicada a correções de Gold, junto com
a investigação do `event_log` (189.984 → 161.842 eventos) já registrada
na seção de SNA.

**Decisão:** não filtrar os 38 casos do resultado agora, qualquer correção
na origem dos dados deve refletir automaticamente aqui, sem necessidade de
ajustar o filtro depois. O resultado da #3 deve ser lido com essa ressalva
até a correção.